# 03 — Term Life Actuarial Calculations

Calculate 20-year death probabilities and individual-policy EPVs using your existing actuarial helper functions.

## Project setup

Run this notebook from the repository root or from the `notebooks/` directory. All paths are resolved to the repository root. This project uses the `src/` helper modules already present in your repository; the original report did not include their source code.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent
if not (project_root / "src").is_dir():
    raise FileNotFoundError(
        "Project root not found. Open the repository in VS Code and run the notebook from the repository root or notebooks/."
    )
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
report_results_dir = project_root / "results"
report_results_dir.mkdir(parents=True, exist_ok=True)
figures_dir = project_root / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
clean_path = project_root / "data" / "processed" / "ssa_mortality_clean.csv"
if not clean_path.is_file():
    raise FileNotFoundError(
        "Run notebook 01 first to generate data/processed/ssa_mortality_clean.csv"
    )
mortality = pd.read_csv(clean_path)
print("Loaded", len(mortality), "mortality records")

entry_age = 40
term = 20


## Term survival and death probabilities

In [ ]:
from src.life_math import term_death_probabilities

probability_summary = []

for sex in ["Male", "Female"]:
    _, survival_prob = term_death_probabilities(
        mortality,
        sex=sex,
        entry_age=entry_age,
        term=term
    )

    probability_summary.append({
        "Mortality schedule": sex,
        "Probability of survival": f"{survival_prob:.4%}",
        "Probability of death": f"{1 - survival_prob:.4%}"
    })

probability_df = pd.DataFrame(probability_summary)

display(probability_df.set_index("Mortality schedule"))

## Expected present value of death benefits

In [ ]:
from src.life_math import (
    term_death_probabilities,
    term_epv
)

entry_age = 40
term = 20
benefit = 100_000
interest_rate = 0.03

actuarial_results = []
death_tables = {}

from src.life_math import portfolio_moments

for sex in ["Male", "Female"]:

    death_table, survival_prob = (
        term_death_probabilities(
            mortality,
            sex=sex,
            entry_age=entry_age,
            term=term
        )
    )

    epv = term_epv(
        death_table,
        benefit=benefit,
        interest_rate=interest_rate
    )

    death_tables[sex] = death_table

    actuarial_results.append({
        "Sex": sex,
        "20-year survival probability": survival_prob,
        "20-year death probability": 1 - survival_prob,
        "Expected present value ($)": epv
    })

results_df = pd.DataFrame(actuarial_results)

display(results_df.round(6))

## Export actuarial tables

In [ ]:
def write_report_table(frame, filename, *, include_index=False, floatfmt=None):
    """Write an output table that the code-free Quarto report can include."""
    kwargs = {"index": include_index}
    if floatfmt is not None:
        kwargs["floatfmt"] = floatfmt
    (report_results_dir / filename).write_text(
        frame.to_markdown(**kwargs) + "\n", encoding="utf-8"
    )

prob_report = probability_df.rename(columns={
    "Mortality schedule":"Mortality schedule"
})
write_report_table(prob_report,"06_term_death_probabilities.qmd")
write_report_table(results_df.round(6),"07_individual_epv.qmd")
print("Updated actuarial tables.")
